# Round16 Colab Multisource Top3 Diagnostic

This notebook isolates the Round16 multisource top3 evidence selection step. It is intentionally different from `round16_colab_full_pipeline.ipynb`:

- `round16_colab_full_pipeline.ipynb` regenerates a lightweight raw-data pipeline and trains models in the notebook.
- This notebook measures the archived multisource fusion path from ranked source JSON files.

The goal is to verify that top3 fusion/selection itself is not the runtime bottleneck. It requires the ranked source artifacts under `outputs/` to be available, or uploaded with the same relative paths. It does not use saved model checkpoints.

In [1]:
from pathlib import Path
import csv
import json
import math
import os
import platform
import subprocess
import sys
import time
from collections import Counter, defaultdict
from itertools import product

START_WALL = time.perf_counter()
TIMINGS = {}

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'train-claims.json').exists() or (candidate / 'agent_docs').exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

def timed(name):
    class Timer:
        def __enter__(self):
            self.start = time.perf_counter()
            return self
        def __exit__(self, exc_type, exc, tb):
            TIMINGS[name] = time.perf_counter() - self.start
    return Timer()

def read_json(path):
    path = Path(path)
    with path.open(encoding='utf-8') as f:
        return json.load(f)

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def path_size_mb(path):
    return Path(path).stat().st_size / (1024 * 1024) if Path(path).exists() else None

print('Python', platform.python_version())
print('PROJECT_ROOT', PROJECT_ROOT)
print('CWD', Path.cwd())

Python 3.12.12
PROJECT_ROOT /mnt/a/code/NLP/A3
CWD /mnt/a/code/NLP/A3


In [2]:
# Data/source paths. Keep these paths if uploading artifacts to Colab.
DATA_DIR = Path('data') if Path('data/train-claims.json').exists() else Path('/content/data')
if not (DATA_DIR / 'train-claims.json').exists() and Path('/content/train-claims.json').exists():
    DATA_DIR = Path('/content')

OUTPUT_DIR = Path('outputs/round16_colab_multisource_top3_local')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CLAIMS_PATH = DATA_DIR / 'train-claims.json'
DEV_CLAIMS_PATH = DATA_DIR / 'dev-claims.json'

LABEL_SOURCE_PATH = Path('outputs/round15/recommended/top3_submission.json')
SOURCE_SPECS = [
    ('binary', Path('outputs/round16/top3_binary_selector_e3_k20_n80/dev_binary_selector_ranked.json')),
    ('r15', Path('outputs/round15/rrf_top3_selector/ranked.json')),
    ('a_fixed', Path('outputs/round16/branch_a_requirement/fixed_rrf_claim_key_ranked_top4500.json')),
    ('s22q', Path('outputs/round16/baseline/s22_small_qprefix_top100.json')),
]

# Fixed configs recorded in round16 docs.
TOP3_FIXED = {
    'top_k': 3,
    'rrf_k': 20.0,
    'cap': 20,
    'weights': {'binary': 2.0, 'r15': 0.5, 'a_fixed': 0.5},
}
TOP10_FIXED = {
    'top_k': 10,
    'rrf_k': 20.0,
    'cap': 50,
    'weights': {'binary': 1.0, 'r15': 0.5, 'a_fixed': 0.5, 's22q': 0.5},
}

RUN_TOP3_SWEEP = True
TOP3_RRF_K_VALUES = [1.0, 5.0, 10.0, 20.0, 60.0]
TOP3_WEIGHT_GRID = {
    'binary': [0.25, 0.5, 1.0, 1.5, 2.0, 3.0],
    'r15': [0.5, 1.0, 1.5, 2.0],
    's22q': [0.0, 0.5, 1.0, 1.5],
    'a_fixed': [0.0, 0.5, 1.0, 1.5],
}
TOP3_SWEEP_CAP = 20

print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('RUN_TOP3_SWEEP =', RUN_TOP3_SWEEP)

DATA_DIR = data
OUTPUT_DIR = outputs/round16_colab_multisource_top3_local
RUN_TOP3_SWEEP = True


In [3]:
def hardware_summary():
    summary = {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'cpu_count_logical': os.cpu_count(),
    }
    try:
        import psutil
        summary['ram_gb'] = round(psutil.virtual_memory().total / (1024 ** 3), 2)
    except Exception as exc:
        summary['ram_gb_error'] = str(exc)
    try:
        import torch
        summary['torch'] = torch.__version__
        summary['cuda_available'] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            summary['gpu_name'] = torch.cuda.get_device_name(0)
            props = torch.cuda.get_device_properties(0)
            summary['gpu_memory_gb'] = round(props.total_memory / (1024 ** 3), 2)
    except Exception as exc:
        summary['torch_error'] = str(exc)
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
            text=True, stderr=subprocess.STDOUT, timeout=5,
        ).strip()
        summary['nvidia_smi'] = out
    except Exception as exc:
        summary['nvidia_smi_error'] = str(exc)
    return summary

HARDWARE = hardware_summary()
print(json.dumps(HARDWARE, indent=2))

{
  "python": "3.12.12",
  "platform": "Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "cpu_count_logical": 8,
  "ram_gb": 29.38,
  "torch": "2.11.0+cu130",
  "cuda_available": true,
  "gpu_name": "NVIDIA GeForce RTX 5070 Ti",
  "gpu_memory_gb": 15.92,
  "nvidia_smi": "NVIDIA GeForce RTX 5070 Ti, 16303 MiB, 591.86"
}


In [4]:
LABELS = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']

def majority_label(claims):
    counts = Counter(claim.get('claim_label') for claim in claims.values() if claim.get('claim_label'))
    return counts.most_common(1)[0][0] if counts else 'SUPPORTS'

def normalize_ranked(data):
    ranked = {}
    for claim_id, value in data.items():
        rows = []
        if isinstance(value, dict) and 'evidences' in value:
            rows = [
                {'evidence_id': evidence_id, 'rank': rank, 'score': 1.0 / rank}
                for rank, evidence_id in enumerate(value.get('evidences', []), start=1)
            ]
        elif isinstance(value, list):
            for rank, item in enumerate(value, start=1):
                if isinstance(item, dict):
                    evidence_id = item.get('evidence_id')
                    item_rank = int(item.get('rank', rank))
                    score = float(item.get('score', item.get('fusion_score', 0.0)))
                else:
                    evidence_id = str(item)
                    item_rank = rank
                    score = 1.0 / rank
                if evidence_id:
                    rows.append({'evidence_id': evidence_id, 'rank': item_rank, 'score': score})
        rows.sort(key=lambda row: (row['rank'], -row.get('score', 0.0), row['evidence_id']))
        seen = set()
        unique = []
        for row in rows:
            if row['evidence_id'] in seen:
                continue
            seen.add(row['evidence_id'])
            unique.append(row)
        ranked[claim_id] = unique
    return ranked

def fuse(claim_ids, sources, weights, rrf_k, cap):
    fused = {}
    for claim_id in claim_ids:
        scores = defaultdict(float)
        source_parts = defaultdict(dict)
        for name, source in sources.items():
            weight = float(weights.get(name, 0.0))
            if weight <= 0:
                continue
            for idx, item in enumerate(source.get(claim_id, [])[:cap], start=1):
                evidence_id = item['evidence_id']
                rank = int(item.get('rank', idx))
                scores[evidence_id] += weight / (rrf_k + rank)
                source_parts[evidence_id][f'{name}_rank'] = rank
        rows = [
            {'evidence_id': evidence_id, 'score': float(score), **source_parts[evidence_id]}
            for evidence_id, score in scores.items()
        ]
        rows.sort(key=lambda row: (-row['score'], row['evidence_id']))
        for rank, row in enumerate(rows, start=1):
            row['rank'] = rank
        fused[claim_id] = rows
    return fused

def predictions_from_ranked(claims, ranked, label_source, default_label, top_k):
    return {
        claim_id: {
            'claim_text': claim['claim_text'],
            'claim_label': label_source.get(claim_id, {}).get('claim_label', default_label),
            'evidences': [item['evidence_id'] for item in ranked.get(claim_id, [])[:top_k]],
        }
        for claim_id, claim in claims.items()
    }

def assignment_metrics(claims, predictions):
    f_scores = []
    accuracies = []
    for claim_id, claim in sorted(claims.items()):
        prediction = predictions.get(claim_id, {})
        if 'claim_label' not in prediction or 'evidences' not in prediction:
            continue
        accuracies.append(1.0 if prediction['claim_label'] == claim.get('claim_label') else 0.0)
        predicted_ids = prediction.get('evidences', [])
        evidence_f = 0.0
        if isinstance(predicted_ids, list) and predicted_ids:
            gold = set(claim.get('evidences', []))
            correct = len(gold & set(predicted_ids))
            if correct > 0 and gold:
                recall = correct / len(gold)
                precision = correct / len(predicted_ids)
                evidence_f = 2 * precision * recall / (precision + recall)
        f_scores.append(evidence_f)
    mean_f = sum(f_scores) / len(f_scores) if f_scores else 0.0
    mean_acc = sum(accuracies) / len(accuracies) if accuracies else 0.0
    harmonic = 0.0 if mean_f == 0.0 and mean_acc == 0.0 else 2 * mean_f * mean_acc / (mean_f + mean_acc)
    return {'retrieval_f_score': mean_f, 'claim_accuracy': mean_acc, 'harmonic_mean': harmonic}

def confusion_rows(claims, predictions):
    rows = []
    for claim_id, claim in claims.items():
        gold = set(claim.get('evidences', []))
        predicted_ids = predictions.get(claim_id, {}).get('evidences', [])
        predicted = set(predicted_ids)
        rows.append({
            'claim_id': claim_id,
            'claim_label': claim.get('claim_label'),
            'tp': len(gold & predicted),
            'fp': len(predicted - gold),
            'fn': len(gold - predicted),
            'gold_count': len(gold),
            'predicted_count': len(predicted_ids),
        })
    return rows

def aggregate_confusion(rows):
    tp = sum(row['tp'] for row in rows)
    fp = sum(row['fp'] for row in rows)
    fn = sum(row['fn'] for row in rows)
    claims = len(rows)
    hit_any = sum(1 for row in rows if row['tp'] > 0)
    all_gold = sum(1 for row in rows if row['fn'] == 0)
    return {
        'tp': tp, 'fp': fp, 'fn': fn,
        'precision': tp / (tp + fp) if tp + fp else 0.0,
        'micro_recall': tp / (tp + fn) if tp + fn else 0.0,
        'hit_any': hit_any / claims if claims else 0.0,
        'all_gold': all_gold / claims if claims else 0.0,
    }

def macro_recall(claims, predictions):
    recalls = []
    for claim_id, claim in claims.items():
        gold = set(claim.get('evidences', []))
        if not gold:
            continue
        predicted = set(predictions.get(claim_id, {}).get('evidences', []))
        recalls.append(len(gold & predicted) / len(gold))
    return sum(recalls) / len(recalls) if recalls else 0.0

def metric_row(name, claims, predictions):
    assignment = assignment_metrics(claims, predictions)
    aggregate = aggregate_confusion(confusion_rows(claims, predictions))
    row = {
        'name': name,
        'macro_recall': macro_recall(claims, predictions),
        'retrieval_f_score': assignment['retrieval_f_score'],
        'claim_accuracy': assignment['claim_accuracy'],
        'harmonic_mean': assignment['harmonic_mean'],
        'micro_recall': aggregate['micro_recall'],
        'precision': aggregate['precision'],
        'hit_any': aggregate['hit_any'],
        'all_gold': aggregate['all_gold'],
    }
    for label in LABELS:
        subset = {cid: c for cid, c in claims.items() if c.get('claim_label') == label}
        row[f'{label.lower()}_macro_recall'] = macro_recall(subset, predictions)
    return row

In [5]:
required_paths = [TRAIN_CLAIMS_PATH, DEV_CLAIMS_PATH, LABEL_SOURCE_PATH] + [path for _, path in SOURCE_SPECS]
missing = [str(path) for path in required_paths if not Path(path).exists()]
if missing:
    print('Missing required files:')
    for item in missing:
        print(' -', item)
    raise FileNotFoundError('Upload/copy the missing source artifacts before running this diagnostic notebook.')

with timed('load_claims_and_label_source'):
    train_claims = read_json(TRAIN_CLAIMS_PATH)
    dev_claims = read_json(DEV_CLAIMS_PATH)
    label_source = read_json(LABEL_SOURCE_PATH)
    default_label = majority_label(train_claims)

source_file_stats = []
sources = {}
with timed('load_and_normalize_sources_total'):
    for name, path in SOURCE_SPECS:
        start = time.perf_counter()
        raw = read_json(path)
        sources[name] = normalize_ranked(raw)
        elapsed = time.perf_counter() - start
        source_file_stats.append({
            'name': name,
            'path': str(path),
            'size_mb': round(path_size_mb(path), 3),
            'claims': len(sources[name]),
            'load_normalize_seconds': elapsed,
        })

print('train claims:', len(train_claims))
print('dev claims:', len(dev_claims))
print('default label:', default_label)
print(json.dumps(source_file_stats, indent=2))

train claims: 1228
dev claims: 154
default label: SUPPORTS
[
  {
    "name": "binary",
    "path": "outputs/round16/top3_binary_selector_e3_k20_n80/dev_binary_selector_ranked.json",
    "size_mb": 0.929,
    "claims": 154,
    "load_normalize_seconds": 0.014187782000135485
  },
  {
    "name": "r15",
    "path": "outputs/round15/rrf_top3_selector/ranked.json",
    "size_mb": 4.987,
    "claims": 154,
    "load_normalize_seconds": 0.052307420000033744
  },
  {
    "name": "a_fixed",
    "path": "outputs/round16/branch_a_requirement/fixed_rrf_claim_key_ranked_top4500.json",
    "size_mb": 117.023,
    "claims": 154,
    "load_normalize_seconds": 1.1989243819998592
  },
  {
    "name": "s22q",
    "path": "outputs/round16/baseline/s22_small_qprefix_top100.json",
    "size_mb": 0.402,
    "claims": 154,
    "load_normalize_seconds": 0.030965812999966147
  }
]


In [6]:
def run_fixed_config(config, name):
    with timed(f'{name}_fuse_seconds'):
        ranked = fuse(dev_claims.keys(), sources, config['weights'], config['rrf_k'], config['cap'])
    with timed(f'{name}_prediction_metric_seconds'):
        predictions = predictions_from_ranked(dev_claims, ranked, label_source, default_label, config['top_k'])
        metrics = metric_row(name, dev_claims, predictions)
    metrics.update({
        'top_k': config['top_k'],
        'rrf_k': config['rrf_k'],
        'cap': config['cap'],
        'weights': '|'.join(f'{k}:{v:g}' for k, v in config['weights'].items() if v > 0),
    })
    write_json(OUTPUT_DIR / f'{name}_ranked.json', ranked)
    write_json(OUTPUT_DIR / f'{name}_predictions.json', predictions)
    return metrics

top3_fixed_metrics = run_fixed_config(TOP3_FIXED, 'top3_fixed_multisource')
top10_fixed_metrics = run_fixed_config(TOP10_FIXED, 'top10_fixed_multisource')

print('TOP3 FIXED')
print(json.dumps(top3_fixed_metrics, indent=2))
print('TOP10 FIXED')
print(json.dumps(top10_fixed_metrics, indent=2))

TOP3 FIXED
{
  "name": "top3_fixed_multisource",
  "macro_recall": 0.3042207792207792,
  "retrieval_f_score": 0.2679499072356215,
  "claim_accuracy": 0.44155844155844154,
  "harmonic_mean": 0.3335141683837507,
  "micro_recall": 0.2606924643584521,
  "precision": 0.27705627705627706,
  "hit_any": 0.5974025974025974,
  "all_gold": 0.12987012987012986,
  "supports_macro_recall": 0.39362745098039215,
  "refutes_macro_recall": 0.19444444444444445,
  "not_enough_info_macro_recall": 0.175609756097561,
  "disputed_macro_recall": 0.42407407407407405,
  "top_k": 3,
  "rrf_k": 20.0,
  "cap": 20,
  "weights": "binary:2|r15:0.5|a_fixed:0.5"
}
TOP10 FIXED
{
  "name": "top10_fixed_multisource",
  "macro_recall": 0.4573593073593074,
  "retrieval_f_score": 0.19439262036664634,
  "claim_accuracy": 0.44155844155844154,
  "harmonic_mean": 0.26994436408274813,
  "micro_recall": 0.4134419551934827,
  "precision": 0.1318181818181818,
  "hit_any": 0.7662337662337663,
  "all_gold": 0.22727272727272727,
  "supp

In [7]:
top3_sweep_best = None
top3_sweep_rows = []

if RUN_TOP3_SWEEP:
    names = list(TOP3_WEIGHT_GRID)
    grids = [TOP3_WEIGHT_GRID[name] for name in names]
    with timed('top3_sweep_seconds'):
        for rrf_k in TOP3_RRF_K_VALUES:
            for values in product(*grids):
                weights = dict(zip(names, values))
                if sum(weights.values()) <= 0:
                    continue
                ranked = fuse(dev_claims.keys(), sources, weights, rrf_k, TOP3_SWEEP_CAP)
                predictions = predictions_from_ranked(dev_claims, ranked, label_source, default_label, 3)
                row = metric_row('top3_sweep', dev_claims, predictions)
                row.update({
                    'top_k': 3,
                    'rrf_k': rrf_k,
                    'cap': TOP3_SWEEP_CAP,
                    'weights': '|'.join(f'{k}:{v:g}' for k, v in weights.items() if v > 0),
                })
                top3_sweep_rows.append(row)
                key = (row['macro_recall'], row['retrieval_f_score'])
                if top3_sweep_best is None or key > top3_sweep_best[0]:
                    top3_sweep_best = (key, row, predictions, ranked)
    top3_sweep_rows.sort(key=lambda row: (-row['macro_recall'], -row['retrieval_f_score']))
    write_csv(OUTPUT_DIR / 'top3_sweep_top100.csv', top3_sweep_rows[:100])
    write_json(OUTPUT_DIR / 'top3_sweep_best_predictions.json', top3_sweep_best[2])
    write_json(OUTPUT_DIR / 'top3_sweep_best_ranked.json', top3_sweep_best[3])
    print('TOP3 SWEEP BEST')
    print(json.dumps(top3_sweep_best[1], indent=2))
    print('sweep rows:', len(top3_sweep_rows))
else:
    print('Top3 sweep disabled.')

TOP3 SWEEP BEST
{
  "name": "top3_sweep",
  "macro_recall": 0.3042207792207792,
  "retrieval_f_score": 0.2679499072356215,
  "claim_accuracy": 0.44155844155844154,
  "harmonic_mean": 0.3335141683837507,
  "micro_recall": 0.2606924643584521,
  "precision": 0.27705627705627706,
  "hit_any": 0.5974025974025974,
  "all_gold": 0.12987012987012986,
  "supports_macro_recall": 0.39362745098039215,
  "refutes_macro_recall": 0.19444444444444445,
  "not_enough_info_macro_recall": 0.175609756097561,
  "disputed_macro_recall": 0.42407407407407405,
  "top_k": 3,
  "rrf_k": 20.0,
  "cap": 20,
  "weights": "binary:2|r15:0.5|a_fixed:0.5"
}
sweep rows: 1920


In [8]:
# Lightweight local benchmark for the pure Python RRF loop.
def microbenchmark_rrf(iterations=200):
    claim_ids = list(dev_claims.keys())
    start = time.perf_counter()
    for _ in range(iterations):
        fuse(claim_ids, sources, TOP3_FIXED['weights'], TOP3_FIXED['rrf_k'], TOP3_FIXED['cap'])
    elapsed = time.perf_counter() - start
    return {
        'iterations': iterations,
        'seconds': elapsed,
        'fixed_top3_fusions_per_second': iterations / elapsed if elapsed else None,
    }

with timed('rrf_microbenchmark_seconds'):
    RRF_MICROBENCH = microbenchmark_rrf(iterations=200)
print(json.dumps(RRF_MICROBENCH, indent=2))

{
  "iterations": 200,
  "seconds": 0.5956707749996895,
  "fixed_top3_fusions_per_second": 335.75593833708604
}


In [9]:
# Crude Colab estimate for this diagnostic notebook.
# The actual Colab runtime depends on assigned CPU, disk throughput, and VM load; Google says these resources vary over time.
COLAB_CPU_SLOWDOWN_RANGE = (2.0, 5.0)  # top3 fusion is CPU/JSON dominated, not GPU dominated.
local_total_so_far = time.perf_counter() - START_WALL
colab_estimate_seconds = {
    'low': local_total_so_far * COLAB_CPU_SLOWDOWN_RANGE[0],
    'high': local_total_so_far * COLAB_CPU_SLOWDOWN_RANGE[1],
}
print('local_total_so_far_seconds =', round(local_total_so_far, 3))
print('colab_estimate_seconds =', {k: round(v, 1) for k, v in colab_estimate_seconds.items()})

local_total_so_far_seconds = 11.591
colab_estimate_seconds = {'low': 23.2, 'high': 58.0}


In [10]:
summary = {
    'notebook': 'round16_colab_multisource_top3_diagnostic.ipynb',
    'purpose': 'Isolate archived multisource top3/top10 RRF selection runtime from ranked source JSON artifacts.',
    'hardware': HARDWARE,
    'source_file_stats': source_file_stats,
    'timings_seconds': {k: round(v, 6) for k, v in TIMINGS.items()},
    'total_wall_seconds': round(time.perf_counter() - START_WALL, 6),
    'top3_fixed_metrics': top3_fixed_metrics,
    'top10_fixed_metrics': top10_fixed_metrics,
    'top3_sweep_best_metrics': top3_sweep_best[1] if top3_sweep_best else None,
    'rrf_microbenchmark': RRF_MICROBENCH,
    'colab_estimate_seconds': colab_estimate_seconds,
    'colab_estimate_note': 'CPU/JSON dominated diagnostic. Estimate uses 2x-5x slowdown because Colab CPU/disk varies; measure on actual runtime for final number.',
}
write_json(OUTPUT_DIR / 'run_summary.json', summary)
print('Wrote', OUTPUT_DIR / 'run_summary.json')
print(json.dumps(summary, indent=2))

Wrote outputs/round16_colab_multisource_top3_local/run_summary.json
{
  "notebook": "round16_colab_multisource_top3_diagnostic.ipynb",
  "purpose": "Isolate archived multisource top3/top10 RRF selection runtime from ranked source JSON artifacts.",
  "hardware": {
    "python": "3.12.12",
    "platform": "Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39",
    "cpu_count_logical": 8,
    "ram_gb": 29.38,
    "torch": "2.11.0+cu130",
    "cuda_available": true,
    "gpu_name": "NVIDIA GeForce RTX 5070 Ti",
    "gpu_memory_gb": 15.92,
    "nvidia_smi": "NVIDIA GeForce RTX 5070 Ti, 16303 MiB, 591.86"
  },
  "source_file_stats": [
    {
      "name": "binary",
      "path": "outputs/round16/top3_binary_selector_e3_k20_n80/dev_binary_selector_ranked.json",
      "size_mb": 0.929,
      "claims": 154,
      "load_normalize_seconds": 0.014187782000135485
    },
    {
      "name": "r15",
      "path": "outputs/round15/rrf_top3_selector/ranked.json",
      "size_mb": 4.987,
      "

## Interpretation

If the fixed multisource top3 result is close to `0.304221` macro recall and the total runtime is far below the raw-data full pipeline, then top3 fusion/selection is not the bottleneck. The expensive part is generating or training the upstream ranked sources, especially the binary selector scoring and any dense candidate generation.

## Strict Local Reproduction Result

Executed on 2026-05-08 in conda env `NLP` with `jupyter nbconvert --execute` after clearing notebook outputs and deleting `outputs/round16_colab_multisource_top3_local/`.

Measured external wall time with `/usr/bin/time -p`:

```txt
real 13.04s
user 12.40s
sys 0.78s
```

Fixed multisource top3 result reproduced exactly:

```json
{
  "macro_recall": 0.3042207792207792,
  "retrieval_f_score": 0.2679499072356215,
  "hit_any": 0.5974025974025974,
  "all_gold": 0.12987012987012986,
  "weights": "binary:2|r15:0.5|a_fixed:0.5"
}
```

Top3 fusion itself took `0.00339s`; the full top3 sweep took `8.269s`.